In [1]:
from transformers import AutoTokenizer, Qwen2ForCausalLM, Qwen2_5_VLForConditionalGeneration, Qwen2_5_VLProcessor, BitsAndBytesConfig

model_dir = "modelfile/checkpoint-2312"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",  
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype="float32"  
)

tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(model_dir, trust_remote_code=True, device_map="auto")
processor = Qwen2_5_VLProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct", trust_remote_code=True)

model.eval()

2025-07-15 07:28:16.512141: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752564496.523486 2337740 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752564496.526919 2337740 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1752564496.536734 2337740 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1752564496.536745 2337740 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1752564496.536747 2337740 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


Qwen2_5_VLForConditionalGeneration(
  (model): Qwen2_5_VLModel(
    (visual): Qwen2_5_VisionTransformerPretrainedModel(
      (patch_embed): Qwen2_5_VisionPatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2_5_VLVisionBlock(
          (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
          (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
          (attn): Qwen2_5_VLVisionSdpaAttention(
            (qkv): Linear(in_features=1280, out_features=3840, bias=True)
            (proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): Qwen2_5_VLMLP(
            (gate_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (up_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (down_proj): Linear(in_features=3420, out_features=1280, bias=True)
            (act_fn): Si

In [2]:
import json
import os
import torch
from qwen_vl_utils import process_vision_info

base_video_dir = "/tf/notebook/NFS_dataset/YOLO_with_ReID/results/test_a_n_video"
with open("/tf/notebook/NFS_dataset/YOLO_with_ReID/results/merged_a_n_test.json", "r") as f:
    data = json.load(f)

system_instruction = (
    "You are a video analysis assistant.\n\n"
    "Your task is to analyze a 5-second video and determine whether any abnormal behavior occurs.\n"
    "Treat any instance of physical harm or violence between individuals as an assault situation.\n\n"
    "If abnormal behavior is detected, respond in one sentence that describes:\n"
    "- the type of anomaly (e.g., assault),\n"
    "- the specific action performed (e.g., punching, pushing, kicking, pulling, threatening, throwing, falldown),\n"
    "- the ID of the person who performed the action (subject),\n"
    "- the ID of the person who was targeted (target),\n"
    "- and what specifically happened between them.\n\n"
    "If the video contains no abnormal behavior, simply respond with:\n"
    "\"This situation can be considered as a normal situation.\"\n\n"
    "Use the video frames and tracking data to support your reasoning.\n"
    "Your response must be concise and grammatically correct.\n"
    "---\n"
    "Examples:\n"
    "1. In an assault situation, ID2 who is wearing a black hoodie and blue jeans is punching ID1 who is wearing a white shirt and black jeans.\n"
    "2. In an assault situation, ID1 who is wearing a red jacket and white pants is kicking ID3 who is wearing a black cardigan black pants.\n"
    "3. This situation can be considered as a normal situation.\n"
)

for i, entry in enumerate(data):
    video_filename = entry["video"]
    video_path = os.path.join(base_video_dir, video_filename)
    
    conversations = entry["conversations"]
    prompt = next((c["value"] for c in conversations if c["from"] == "human"), None)
    if prompt is None:
        continue

    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": [
            {"type": "video", "video": video_path, "fps": 1.0},
            {"type": "text", "text": prompt}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda:0")

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=64)

    result = outputs[:, inputs.input_ids.shape[-1]:]

    response = processor.batch_decode(result, skip_special_tokens=True)[0]
    print(f"🤖 video: {video_path}")
    print(f"🤖 Answer: {response}")
    print()

qwen-vl-utils using torchvision to read video.
/usr/local/lib/python3.11/dist-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(
Unused or unrecognized kwargs: return_tensors, fps.


🤖 video: /tf/notebook/NFS_dataset/YOLO_with_ReID/results/test_a_n_video/21-3_cam01_normal01_place09_day_spring_26.mp4
🤖 Answer: In an assault situation, ID6 who is wearing a white shirt and black pants is kicking ID1 who is wearing a red shirt and black pants.



Unused or unrecognized kwargs: return_tensors, fps.


🤖 video: /tf/notebook/NFS_dataset/YOLO_with_ReID/results/test_a_n_video/7-3_cam02_normal01_place04_day_spring_5.mp4
🤖 Answer: This situation can be considered as a normal situation



Unused or unrecognized kwargs: return_tensors, fps.


...

🤖 video: /tf/notebook/NFS_dataset/YOLO_with_ReID/results/test_a_n_video/4-5_cam02_assault01_place01_day_summer_person1_pushing_seg2.mp4
🤖 Answer: In an assault situation, ID1 who is wearing a white shirt and blue jeans is kicking ID2 who is wearing a white shirt and blue jeans.



Unused or unrecognized kwargs: return_tensors, fps.


🤖 video: /tf/notebook/NFS_dataset/YOLO_with_ReID/results/test_a_n_video/4-3_cam02_assault01_place01_day_summer_person1_pulling_seg4.mp4
🤖 Answer: In an assault situation, ID2 who is wearing white shirt and blue jeans is punching ID1 who is wearing white shirt and blue jeans.



Unused or unrecognized kwargs: return_tensors, fps.


🤖 video: /tf/notebook/NFS_dataset/YOLO_with_ReID/results/test_a_n_video/5-1_cam02_assault01_place08_day_spring_person2_threaten_seg5.mp4
🤖 Answer: In an assault situation, ID1 who is wearing black shirt and black pants is punching ID2 who is wearing black shirt and black pants.

🤖 video: /tf/notebook/NFS_dataset/YOLO_with_ReID/results/test_a_n_video/12-4_cam01_assault01_place09_day_spring_person1_throwing_seg1.mp4
🤖 Answer: In an assault situation, ID1 who is wearing checkered shirt and blue pants is kicking ID2 who is wearing checkered shirt and black pants.

